# Interactive Canny + Hough review

Inspired by the EPFL Canny edge-detector demo
(<https://bigwww.epfl.ch/demo/ip/demos/edgeDetector/>), but tied to the
real `concam.detection.detect` call the pipeline runs.

Pick a labeled ROI from `output/validation/detection/2026-04-08/`, then scrub
the Canny / Hough / angle-filter knobs and watch the four-panel figure
update live:

1. **Crop + rotated polygon** — what the pipeline hands to the detector.
2. **Pixel floor** — the masked crop after the adaptive low-percentile floor
   zeroes out low-contrast sky. This is what Canny actually sees.
3. **Canny edges** — masked to the rotated polygon.
4. **Hough overlay** — aligned lines in green, rejected (off-axis) lines in
   red, flight-path vector in orange, picked `pixel_line` in thick green.

A separate cell runs the current knob settings over all 20 labels and shows
AUC + a score histogram, so you can tell whether your manual tweak would
beat the sweep-recommended combo.

Requires `ipywidgets` + `matplotlib`. Install with
`uv sync --extra review` after the optional dep group is added to
`pyproject.toml`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
%pip install av
import av
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import (
    Checkbox, Dropdown, FloatSlider, HBox, IntSlider, Layout, Output, VBox,
    interactive_output,
)

# Make the package importable when running the notebook from notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from concam.config import DetectionConfig, load_config
from concam.detection import detect
from concam.projection import PixelPoint, Rect, rotated_polygon

# --- Pick a batch ---
# Change DATE_DIR to look at April 9 / Oct 19 / batch 2. Labels are optional.
DATE_DIR = REPO_ROOT / "output" / "validation" / "detection" / "2026-04-08"

MANIFEST = json.loads((DATE_DIR / "manifest.json").read_text())
LABELS_PATH = DATE_DIR / "labels.json"
LABEL_BY_IDX: dict[int, str] = {}
if LABELS_PATH.exists():
    LABELS = json.loads(LABELS_PATH.read_text())
    LABEL_BY_IDX = {e["idx"]: e["label"] for e in LABELS["labels"]}

CANDIDATES = [c for c in MANIFEST["candidates"]]
CROPS = {c["idx"]: cv2.imread(str(DATE_DIR / c["roi_png"])) for c in CANDIDATES}

# --- Pre-fetch prev-frame crops so the diff toggle is instant ---
# Mirrors scripts/detection_review_panels.py: open the manifest video, seek to
# frame_idx-1 for each candidate, crop the same padded AABB. If the video is
# smaller than the calibration resolution, bilinearly upscale first (e.g. 720p
# Oct 2025 archive vs 4K calibration).
SITE_CONFIG = load_config(REPO_ROOT / "configs" / "mit_green_building.yaml")
CALIB_W, CALIB_H = (
    int(SITE_CONFIG.calibration.calibration_resolution[0]),
    int(SITE_CONFIG.calibration.calibration_resolution[1]),
)
EXTRACT_PAD = 20

def _crop_padded(frame, roi, pad=EXTRACT_PAD):
    h, w = frame.shape[:2]
    x1 = max(0, int(roi["x"]) - pad)
    y1 = max(0, int(roi["y"]) - pad)
    x2 = min(w, int(roi["x"]) + int(roi["w"]) + pad)
    y2 = min(h, int(roi["y"]) + int(roi["h"]) + pad)
    return frame[y1:y2, x1:x2].copy()

def _fetch_prev_crops():
    video = Path(MANIFEST["video"])
    if not video.exists():
        print(f"  WARN: manifest video missing ({video}), diff toggle will be disabled.")
        return {}
    # Probe video resolution to decide whether to upscale.
    with av.open(str(video)) as probe:
        vs = probe.streams.video[0]
        vw, vh = int(vs.codec_context.width), int(vs.codec_context.height)
    upscale = (CALIB_W, CALIB_H) if (vw, vh) != (CALIB_W, CALIB_H) else None
    if upscale:
        print(f"  Upscaling prev frames {vw}x{vh} -> {CALIB_W}x{CALIB_H} (calibration).")

    container = av.open(str(video))
    out: dict[int, np.ndarray] = {}
    try:
        stream = container.streams.video[0]
        time_base = stream.time_base
        duration_s = float(stream.duration * stream.time_base) if stream.duration else 0.0
        total_frames = (
            int(stream.frames) if stream.frames
            else int(round(duration_s * float(stream.average_rate or 30)))
        )
        targets = sorted({int(c["frame_idx"]) - 1 for c in CANDIDATES if int(c["frame_idx"]) > 0})
        for target_idx in targets:
            target_time_s = (target_idx / total_frames) * duration_s if total_frames else 0.0
            target_pts = int(target_time_s / float(time_base))
            container.seek(target_pts, stream=stream, any_frame=False, backward=True)
            decoded = None
            for frame in container.decode(stream):
                decoded = frame
                if frame.pts is not None and frame.pts >= target_pts:
                    break
            if decoded is None:
                continue
            arr = decoded.to_ndarray(format="bgr24")
            if upscale and (arr.shape[1], arr.shape[0]) != upscale:
                arr = cv2.resize(arr, upscale, interpolation=cv2.INTER_LINEAR)
            for c in CANDIDATES:
                if int(c["frame_idx"]) - 1 == target_idx:
                    out[int(c["idx"])] = _crop_padded(arr, c["roi"])
    finally:
        container.close()
    return out

print(
    f"Loaded {len(CANDIDATES)} ROIs from {DATE_DIR.name} — "
    f"positive={sum(1 for v in LABEL_BY_IDX.values() if v == 'positive')}, "
    f"negative={sum(1 for v in LABEL_BY_IDX.values() if v == 'negative')}, "
    f"unlabeled={len(CANDIDATES) - len(LABEL_BY_IDX)}"
)
print("Decoding prev frames for diff-toggle (one-time, may take a few seconds)...")
PREV_CROPS = _fetch_prev_crops()
print(f"  cached {len(PREV_CROPS)} prev crops")


In [ ]:
# Mirror the sweep's geometry reconstruction so the notebook drives the real
# detector the same way scripts/detection_validation_sweep.py does.
ROI_ALONG_PX = 120
ROI_CROSS_PX = 40

def reconstruct_geometry(cand: dict, crop_shape):
    ch, cw = crop_shape
    roi = cand["roi"]
    full_tl_x = max(0, int(roi["x"]) - EXTRACT_PAD)
    full_tl_y = max(0, int(roi["y"]) - EXTRACT_PAD)
    center = PixelPoint(
        x=float(cand["pixel_x"]) - full_tl_x,
        y=float(cand["pixel_y"]) - full_tl_y,
    )
    path_vec = (float(cand["path_dx"]), float(cand["path_dy"]))
    dummy_cfg = DetectionConfig(
        roi_along_px=ROI_ALONG_PX, roi_cross_px=ROI_CROSS_PX, roi_padding=20,
    )
    poly = rotated_polygon(center, path_vec, dummy_cfg)
    rect = Rect(x=0, y=0, w=cw, h=ch)
    return rect, poly, path_vec, center

def build_config(
    use_adaptive_canny,
    canny_low, canny_high,
    canny_percentile_low, canny_percentile_high,
    canny_low_ratio, canny_min_high,
    hough_threshold, hough_min_line_length, hough_max_line_gap,
    angle_tolerance_deg, long_line_min_px, score_norm_count,
    use_rotated_mask, blur_kernel,
):
    return DetectionConfig(
        score_threshold=0.3,
        canny_low=int(canny_low),
        canny_high=int(canny_high),
        hough_threshold=int(hough_threshold),
        hough_min_line_length=int(hough_min_line_length),
        hough_max_line_gap=int(hough_max_line_gap),
        roi_padding=20,
        roi_along_px=ROI_ALONG_PX,
        roi_cross_px=ROI_CROSS_PX,
        use_adaptive_canny=bool(use_adaptive_canny),
        canny_percentile_low=float(canny_percentile_low),
        canny_percentile_high=float(canny_percentile_high),
        canny_low_ratio=float(canny_low_ratio),
        canny_min_high=int(canny_min_high),
        angle_tolerance_deg=float(angle_tolerance_deg),
        long_line_min_px=float(long_line_min_px),
        score_norm_count=int(score_norm_count),
        use_rotated_mask=bool(use_rotated_mask),
        blur_kernel=int(blur_kernel),
    )

def compute_panels(crop, rect, poly, path_vec, cfg, prev_crop=None):
    """Reproduce the Canny pre-processing steps so we can render intermediate panels.
    The detector itself is still called for the final score / pixel_line — we only
    duplicate the bits Canny sees.
    If prev_crop is provided and shape-matches, applies the same cv2.absdiff the
    detector does so the panels reflect the diff path the live pipeline runs.
    """
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
    base = gray
    if prev_crop is not None:
        prev_gray = cv2.cvtColor(prev_crop, cv2.COLOR_BGR2GRAY) if prev_crop.ndim == 3 else prev_crop
        if prev_gray.shape == gray.shape:
            base = cv2.absdiff(gray, prev_gray)
    if cfg.blur_kernel and cfg.blur_kernel > 1:
        k = int(cfg.blur_kernel) | 1
        base = cv2.GaussianBlur(base, (k, k), 0)
    mask = None
    if cfg.use_rotated_mask:
        mask = np.zeros(base.shape, dtype=np.uint8)
        cv2.fillPoly(mask, [poly.astype(np.int32)], 255)
        masked_values = base[mask > 0]
    else:
        masked_values = base.reshape(-1)
    if cfg.use_adaptive_canny and masked_values.size:
        p_hi = float(np.percentile(masked_values, cfg.canny_percentile_high))
        p_lo = float(np.percentile(masked_values, cfg.canny_percentile_low))
        canny_high = max(int(round(p_hi)), int(cfg.canny_min_high))
        canny_low = max(1, int(round(canny_high * cfg.canny_low_ratio)))
        floor = int(round(p_lo))
    else:
        canny_high = int(cfg.canny_high)
        canny_low = int(cfg.canny_low)
        floor = 0
    crop_for_canny = base.copy()
    if mask is not None:
        crop_for_canny = cv2.bitwise_and(crop_for_canny, crop_for_canny, mask=mask)
    if floor > 0:
        _, crop_for_canny = cv2.threshold(crop_for_canny, floor, 255, cv2.THRESH_TOZERO)
    edges = cv2.Canny(crop_for_canny, canny_low, canny_high)
    if mask is not None:
        edges = cv2.bitwise_and(edges, edges, mask=mask)
    raw = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi/180.0,
        threshold=int(cfg.hough_threshold),
        minLineLength=int(cfg.hough_min_line_length),
        maxLineGap=int(cfg.hough_max_line_gap),
    )
    raw_lines = [] if raw is None else [tuple(int(v) for v in ln[0]) for ln in raw]
    return {
        "gray": gray,
        "blurred": base,
        "mask": mask,
        "canny_low": canny_low,
        "canny_high": canny_high,
        "floor": floor,
        "crop_for_canny": crop_for_canny,
        "edges": edges,
        "raw_lines": raw_lines,
    }


In [ ]:
# --- Controls ---
_full = Layout(width="95%")

cand_choices = [
    (
        f"#{c['idx']:02d}  {c['callsign']}"
        + (f"  [{LABEL_BY_IDX[c['idx']]}]" if c['idx'] in LABEL_BY_IDX else "  [unlabeled]"),
        c["idx"],
    )
    for c in CANDIDATES
]
w_idx = Dropdown(options=cand_choices, description="candidate", layout=_full)

# IMPORTANT: w_use_diff defaults to True so the notebook matches the live
# pipeline (concam/pipeline/stages.py:374-378 always passes prev_frame).
w_use_diff  = Checkbox(value=True, description="use prev-frame diff (matches live pipeline)")
w_use_adapt = Checkbox(value=True, description="adaptive Canny")
w_use_mask  = Checkbox(value=True, description="rotated mask")

w_pct_high = FloatSlider(value=99.0, min=97.0, max=99.9, step=0.1, description="pct_high", layout=_full)
w_pct_low  = FloatSlider(value=96.0, min=80.0, max=99.0, step=0.5, description="pct_low", layout=_full)
w_low_ratio= FloatSlider(value=0.25, min=0.1, max=0.8, step=0.05, description="low_ratio", layout=_full)
w_min_high = IntSlider(value=60, min=10, max=200, step=5, description="min_high", layout=_full)

w_canny_low  = IntSlider(value=50, min=1, max=255, step=1, description="canny_low", layout=_full)
w_canny_high = IntSlider(value=150, min=1, max=400, step=1, description="canny_high", layout=_full)

w_blur = IntSlider(value=3, min=0, max=11, step=1, description="blur_kernel", layout=_full)

w_hough_thr  = IntSlider(value=15, min=5, max=100, step=1, description="hough_thr", layout=_full)
w_hough_minL = IntSlider(value=20, min=5, max=80, step=1, description="hough_minL", layout=_full)
w_hough_gap  = IntSlider(value=5, min=1, max=30, step=1, description="hough_gap", layout=_full)

w_tol   = FloatSlider(value=12.0, min=1.0, max=45.0, step=0.5, description="angle_tol", layout=_full)
w_longL = FloatSlider(value=15.0, min=5.0, max=80.0, step=1.0, description="long_min", layout=_full)
w_norm  = IntSlider(value=6, min=1, max=20, step=1, description="score_norm", layout=_full)

controls = VBox([
    w_idx,
    HBox([w_use_diff, w_use_adapt, w_use_mask, w_blur]),
    HBox([w_pct_high, w_pct_low]),
    HBox([w_low_ratio, w_min_high]),
    HBox([w_canny_low, w_canny_high]),
    HBox([w_hough_thr, w_hough_minL, w_hough_gap]),
    HBox([w_tol, w_longL, w_norm]),
])

out = Output()

def _render(
    idx, use_diff, use_adaptive_canny, use_rotated_mask, blur_kernel,
    canny_percentile_high, canny_percentile_low,
    canny_low_ratio, canny_min_high,
    canny_low, canny_high,
    hough_threshold, hough_min_line_length, hough_max_line_gap,
    angle_tolerance_deg, long_line_min_px, score_norm_count,
):
    crop = CROPS[idx]
    cand = next(c for c in CANDIDATES if c["idx"] == idx)
    label = LABEL_BY_IDX.get(idx, "unlabeled")
    rect, poly, path_vec, _center = reconstruct_geometry(cand, crop.shape[:2])
    cfg = build_config(
        use_adaptive_canny, canny_low, canny_high,
        canny_percentile_low, canny_percentile_high,
        canny_low_ratio, canny_min_high,
        hough_threshold, hough_min_line_length, hough_max_line_gap,
        angle_tolerance_deg, long_line_min_px, score_norm_count,
        use_rotated_mask, blur_kernel,
    )
    prev_crop = PREV_CROPS.get(idx) if use_diff else None
    panels = compute_panels(crop, rect, poly, path_vec, cfg, prev_crop=prev_crop)
    result = detect(crop, rect, cfg, polygon=poly, path_vec=path_vec, prev_frame=prev_crop)

    import math
    path_angle = math.degrees(math.atan2(path_vec[1], path_vec[0])) % 180.0
    tol = float(cfg.angle_tolerance_deg)
    def _aligned(x1, y1, x2, y2):
        a = math.degrees(math.atan2(y2 - y1, x2 - x1)) % 180.0
        d = abs(((a - path_angle + 90.0) % 180.0) - 90.0)
        return d <= tol

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        vis = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        axes[0, 0].imshow(vis)
        poly_closed = np.vstack([poly, poly[:1]])
        axes[0, 0].plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.5)
        cx = float(cand["pixel_x"]) - max(0, cand["roi"]["x"] - EXTRACT_PAD)
        cy = float(cand["pixel_y"]) - max(0, cand["roi"]["y"] - EXTRACT_PAD)
        L = 40.0
        axes[0, 0].plot(
            [cx - L * path_vec[0], cx + L * path_vec[0]],
            [cy - L * path_vec[1], cy + L * path_vec[1]],
            color="#ff6030", lw=1.0, alpha=0.8,
        )
        axes[0, 0].scatter([cx], [cy], c="#ff6030", s=12)
        diff_tag = "  diff:on" if (use_diff and prev_crop is not None) else "  diff:off"
        axes[0, 0].set_title(
            f"#{idx:02d} {cand['callsign']}  [{label}]{diff_tag}\n"
            f"crop {crop.shape[1]}×{crop.shape[0]}  path={path_angle:5.1f}°"
        )
        axes[0, 0].axis("off")

        axes[0, 1].imshow(panels["crop_for_canny"], cmap="gray", vmin=0, vmax=255)
        axes[0, 1].set_title(
            f"post-floor (floor={panels['floor']})  "
            f"canny_low={panels['canny_low']}  canny_high={panels['canny_high']}"
        )
        axes[0, 1].axis("off")

        axes[1, 0].imshow(panels["edges"], cmap="gray")
        axes[1, 0].set_title(f"Canny edges  ({int(panels['edges'].sum() / 255)} edge px)")
        axes[1, 0].axis("off")

        axes[1, 1].imshow(vis)
        axes[1, 1].plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.0)
        n_aligned = n_rejected = 0
        for (x1, y1, x2, y2) in panels["raw_lines"]:
            if _aligned(x1, y1, x2, y2):
                axes[1, 1].plot([x1, x2], [y1, y2], color="#50e050", lw=1.0, alpha=0.9)
                n_aligned += 1
            else:
                axes[1, 1].plot([x1, x2], [y1, y2], color="#e04040", lw=0.8, alpha=0.5)
                n_rejected += 1
        if result.pixel_line is not None:
            x1, y1, x2, y2 = result.pixel_line
            axes[1, 1].plot([x1, x2], [y1, y2], color="#30ff30", lw=2.5)
        axes[1, 1].set_title(
            f"score={result.score:.3f}  aligned={n_aligned}  "
            f"rejected={n_rejected}  long={result.num_long_lines}  "
            f"method={result.method}"
        )
        axes[1, 1].axis("off")
        plt.tight_layout()
        plt.show()

ui_out = interactive_output(_render, {
    "idx": w_idx,
    "use_diff": w_use_diff,
    "use_adaptive_canny": w_use_adapt,
    "use_rotated_mask": w_use_mask,
    "blur_kernel": w_blur,
    "canny_percentile_high": w_pct_high,
    "canny_percentile_low": w_pct_low,
    "canny_low_ratio": w_low_ratio,
    "canny_min_high": w_min_high,
    "canny_low": w_canny_low,
    "canny_high": w_canny_high,
    "hough_threshold": w_hough_thr,
    "hough_min_line_length": w_hough_minL,
    "hough_max_line_gap": w_hough_gap,
    "angle_tolerance_deg": w_tol,
    "long_line_min_px": w_longL,
    "score_norm_count": w_norm,
})
display(controls, out, ui_out)


## Bulk evaluation on the current knob settings

Re-run the cell below after tweaking sliders above to see AUC + a score
histogram over all labeled ROIs. AUC is a Mann-Whitney-U estimate — same one
`scripts/detection_validation_sweep.py` reports.

The sweep's current best was **AUC=1.000** at `detection_threshold=0.083` with
the knobs in `configs/mit_green_building.yaml`. Beating that on this tiny
20-label set is probably overfit — but noticing a degradation is useful.

In [ ]:
def bulk_eval():
    cfg = build_config(
        w_use_adapt.value, w_canny_low.value, w_canny_high.value,
        w_pct_low.value, w_pct_high.value,
        w_low_ratio.value, w_min_high.value,
        w_hough_thr.value, w_hough_minL.value, w_hough_gap.value,
        w_tol.value, w_longL.value, w_norm.value,
        w_use_mask.value, w_blur.value,
    )
    pos, neg, unlabeled = [], [], []
    for c in CANDIDATES:
        crop = CROPS[c["idx"]]
        rect, poly, pv, _ = reconstruct_geometry(c, crop.shape[:2])
        prev = PREV_CROPS.get(c["idx"]) if w_use_diff.value else None
        s = detect(crop, rect, cfg, polygon=poly, path_vec=pv, prev_frame=prev).score
        bucket = LABEL_BY_IDX.get(c["idx"])
        if bucket == "positive":
            pos.append(s)
        elif bucket == "negative":
            neg.append(s)
        else:
            unlabeled.append(s)

    if pos and neg:
        wins = sum(1.0 if p > n else 0.5 if p == n else 0.0 for p in pos for n in neg)
        auc = wins / (len(pos) * len(neg))
        scores = sorted(set(pos + neg))
        cuts = [(a + b) / 2 for a, b in zip(scores[:-1], scores[1:])] or [0.0]
        best_t, best_j = cuts[0], -1.0
        for t in cuts:
            tpr = sum(1 for p in pos if p >= t) / len(pos)
            fpr = sum(1 for n in neg if n >= t) / len(neg)
            if tpr - fpr > best_j:
                best_j, best_t = tpr - fpr, t
    else:
        auc, best_t, best_j = float("nan"), 0.0, 0.0

    fig, ax = plt.subplots(figsize=(9, 3.5))
    edges = np.linspace(0, 1, 21)
    if neg: ax.hist(neg, bins=edges, alpha=0.6, label=f"negative (n={len(neg)})", color="#3c78d8")
    if pos: ax.hist(pos, bins=edges, alpha=0.6, label=f"positive (n={len(pos)})", color="#e06666")
    if unlabeled: ax.hist(unlabeled, bins=edges, alpha=0.4, label=f"unlabeled (n={len(unlabeled)})", color="#888888")
    ax.axvline(best_t, color="k", ls="--", label=f"Youden-J thr={best_t:.3f}")
    diff_tag = "diff:on" if w_use_diff.value else "diff:off"
    pos_med = float(np.median(pos)) if pos else 0.0
    neg_max = float(max(neg)) if neg else 0.0
    ax.set_title(
        f"AUC={auc:.3f}   J={best_j:.3f}   pos_med={pos_med:.3f}   "
        f"neg_max={neg_max:.3f}   ({diff_tag})"
    )
    ax.set_xlabel("detector score")
    ax.set_ylabel("count")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return {"auc": auc, "threshold": best_t, "pos": pos, "neg": neg, "unlabeled": unlabeled}

bulk_eval()


## Export the current knobs as a YAML block

Handy when you've found a setting that looks better than what's in
`configs/mit_green_building.yaml` and want to paste it in.

In [ ]:
print(f"""detection:
  score_threshold: 0.3
  canny_low: {w_canny_low.value}
  canny_high: {w_canny_high.value}
  hough_threshold: {w_hough_thr.value}
  hough_min_line_length: {w_hough_minL.value}
  hough_max_line_gap: {w_hough_gap.value}
  roi_padding: 20
  roi_along_px: {ROI_ALONG_PX}
  roi_cross_px: {ROI_CROSS_PX}
  use_adaptive_canny: {str(w_use_adapt.value).lower()}
  canny_percentile_low: {w_pct_low.value}
  canny_percentile_high: {w_pct_high.value}
  canny_low_ratio: {w_low_ratio.value}
  canny_min_high: {w_min_high.value}
  angle_tolerance_deg: {w_tol.value}
  long_line_min_px: {w_longL.value}
  score_norm_count: {w_norm.value}
  use_rotated_mask: {str(w_use_mask.value).lower()}
  blur_kernel: {w_blur.value}
""")